# ICT4403 Assignment 2 — Autoregressive Text Generation with RNN, BiRNN, LSTM and GRU

**Objective:** Train and compare four recurrent neural network architectures for next-word prediction and sequential text generation.

**Dataset:** Tiny Shakespeare, a compact public-domain text corpus hosted by Andrej Karpathy:
https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

**Reproducibility:** This notebook fixes random seeds, uses the same tokenization, train/validation split, sequence length, embedding size, hidden size, optimizer, batch size, and epoch budget across models. Results are measured at runtime; do not replace the reported metrics with guessed values.

**Compute note:** The default configuration is deliberately small for a CPU laptop. If training is too slow, lower `MAX_TOKENS` or `EPOCHS`, and state the final configuration in your report.


In [ ]:
# If running in a fresh Colab/Jupyter environment, install dependencies first:
# %pip install -q torch requests pandas matplotlib

import os, re, time, math, random, urllib.request
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)


## 1. Dataset and preprocessing

Tiny Shakespeare contains excerpts from Shakespeare's plays. It is a small literary text corpus suitable for a compact language-modelling demonstration. We normalize whitespace, tokenize words and punctuation separately, add an `<unk>` token for unseen validation tokens, and construct fixed-length context/next-token pairs.

Dataset source: https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare
Raw text file: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = "tinyshakespeare.txt"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Keep words and punctuation as separate tokens; collapse repeated whitespace.
raw_text = re.sub(r"\\s+", " ", raw_text).strip()
tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\\w\\s]", raw_text.lower())

print("Characters:", len(raw_text))
print("Tokens:", len(tokens))
print("Sample:", " ".join(tokens[:45]))


In [ ]:
# Fixed-size experiment for reasonable runtime on modest hardware.
MAX_TOKENS = 60000      # set None to use all tokens (slower)
SEQ_LEN = 12
BATCH_SIZE = 128
EMBED_DIM = 48
HIDDEN_DIM = 64
EPOCHS = 3
LEARNING_RATE = 1e-3

if MAX_TOKENS is not None:
    tokens = tokens[:MAX_TOKENS]

split = int(len(tokens) * 0.90)
train_tokens = tokens[:split]
valid_tokens = tokens[split:]

# Build vocabulary from training text only to avoid validation leakage.
vocab = ["<pad>", "<unk>"] + sorted(set(train_tokens) - {"<pad>", "<unk>"})
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}
UNK_ID = stoi["<unk>"]
PAD_ID = stoi["<pad>"]

train_ids = [stoi.get(t, UNK_ID) for t in train_tokens]
valid_ids = [stoi.get(t, UNK_ID) for t in valid_tokens]

class NextWordDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len
    def __len__(self):
        return max(0, len(self.ids) - self.seq_len)
    def __getitem__(self, idx):
        return self.ids[idx:idx+self.seq_len], self.ids[idx+self.seq_len]

train_ds = NextWordDataset(train_ids, SEQ_LEN)
valid_ds = NextWordDataset(valid_ids, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"Vocabulary size: {len(vocab):,}")
print(f"Train next-word examples: {len(train_ds):,}")
print(f"Validation next-word examples: {len(valid_ds):,}")
print(f"Train/validation token split: {len(train_tokens)}/{len(valid_tokens)}")


## 2. Define comparable models

The Bidirectional RNN uses both directions within the supplied context window. For strict online forecasting, a bidirectional encoder is not causal within a context; however, when predicting the word after a fully observed seed/context, all context words are already available. All models return a vocabulary-sized next-word distribution.


In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, kind, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.kind = kind
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        if kind == "Simple RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity="tanh")
            out_dim = hidden_dim
        elif kind == "Bidirectional RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, bidirectional=True, nonlinearity="tanh")
            out_dim = hidden_dim * 2
        elif kind == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
            out_dim = hidden_dim
        elif kind == "GRU":
            self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
            out_dim = hidden_dim
        else:
            raise ValueError(kind)
        self.dropout = nn.Dropout(0.15)
        self.fc = nn.Linear(out_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        return self.fc(self.dropout(out[:, -1, :]))

MODEL_NAMES = ["Simple RNN", "Bidirectional RNN", "LSTM", "GRU"]


In [ ]:
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)
            n = y.size(0)
            total_loss += loss.item() * n
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_n += n
    mean_loss = total_loss / max(total_n, 1)
    return {"loss": mean_loss, "perplexity": math.exp(min(mean_loss, 20)), 
            "accuracy": total_correct / max(total_n, 1)}

def train_one(kind):
    # Reset seed before each model so the comparison is more reproducible.
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = LanguageModel(kind, len(vocab), EMBED_DIM, HIDDEN_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    history = []
    start = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += loss.item() * y.size(0)
            seen += y.size(0)
        val = evaluate(model, valid_loader)
        history.append({"epoch": epoch, "train_loss": running_loss / max(seen, 1),
                        "val_loss": val["loss"], "val_perplexity": val["perplexity"],
                        "val_accuracy": val["accuracy"]})
        print(f"{kind} | epoch {epoch}/{EPOCHS} | train loss {history[-1]['train_loss']:.4f} "
              f"| val loss {val['loss']:.4f} | val ppl {val['perplexity']:.2f} "
              f"| val acc {val['accuracy']:.3f}")

    elapsed = time.perf_counter() - start
    final = evaluate(model, valid_loader)
    return model, history, {"model": kind, **final, "training_time_seconds": elapsed}

models, histories, summary_rows = {}, {}, []
for name in MODEL_NAMES:
    model, hist, row = train_one(name)
    models[name] = model
    histories[name] = hist
    summary_rows.append(row)

results = pd.DataFrame(summary_rows).sort_values("perplexity")
results


## 3. Quantitative comparison

Cross-entropy loss and perplexity are evaluated on held-out next-word examples. Accuracy is the proportion of validation targets predicted exactly. Training time is measured wall-clock time and depends on hardware, so results can differ between runs.


In [ ]:
display(results.round(4))
results.to_csv("model_comparison.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
results.set_index("model")["loss"].plot(kind="bar", ax=axes[0], title="Validation loss")
results.set_index("model")["perplexity"].plot(kind="bar", ax=axes[1], title="Validation perplexity")
results.set_index("model")["accuracy"].plot(kind="bar", ax=axes[2], title="Validation accuracy")
for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("model_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

history_rows = []
for name, hist in histories.items():
    for item in hist:
        history_rows.append({"model": name, **item})
history_df = pd.DataFrame(history_rows)
history_df.to_csv("training_history.csv", index=False)

plt.figure(figsize=(8, 5))
for name, hist in histories.items():
    plt.plot([x["epoch"] for x in hist], [x["val_loss"] for x in hist], marker="o", label=name)
plt.xlabel("Epoch"); plt.ylabel("Validation loss"); plt.title("Validation loss by epoch")
plt.legend(); plt.tight_layout()
plt.savefig("validation_loss.png", dpi=160, bbox_inches="tight")
plt.show()


## 4. Autoregressive generation and temperature

At each step, the model predicts a distribution for the next token. Temperature rescales logits before softmax: lower temperatures concentrate probability on likely words; higher temperatures flatten the distribution and usually increase variation and risk of incoherence.


In [ ]:
def detokenize(words):
    text = " ".join(words)
    # Attach common punctuation to the preceding word.
    text = re.sub(r"\\s+([,.;:!?])", r"\\1", text)
    text = re.sub(r"\\s+'", "'", text)
    return text

def generate(model, seed_text, temperature=1.0, max_new_tokens=60):
    model.eval()
    seed_tokens = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\\w\\s]", seed_text.lower())
    if not seed_tokens:
        raise ValueError("Seed text must contain at least one token.")
    ids = [stoi.get(t, UNK_ID) for t in seed_tokens]
    generated = list(seed_tokens)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = ids[-SEQ_LEN:]
            if len(context) < SEQ_LEN:
                context = [PAD_ID] * (SEQ_LEN - len(context)) + context
            x = torch.tensor([context], dtype=torch.long, device=DEVICE)
            logits = model(x)[0]
            logits = logits / max(float(temperature), 1e-4)
            # Never sample the padding token.
            logits[PAD_ID] = -1e9
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            ids.append(next_id)
            generated.append(itos[next_id])
    return detokenize(generated)

SEEDS = [
    "the king said",
    "to be or not",
    "my lord, the"
]
TEMPERATURES = [0.5, 1.0, 1.5]

generation_rows = []
for model_name, model in models.items():
    for seed in SEEDS:
        for temp in TEMPERATURES:
            sample = generate(model, seed, temperature=temp, max_new_tokens=45)
            generation_rows.append({"model": model_name, "seed": seed,
                                    "temperature": temp, "generated_text": sample})
generation_df = pd.DataFrame(generation_rows)
generation_df.to_csv("generated_samples.csv", index=False)
display(generation_df.head(12))


## 5. Qualitative analysis worksheet

Review `generated_samples.csv` and record observations grounded in the outputs you actually see. Do not claim that a model is more coherent or diverse unless the samples support that conclusion.

Suggested dimensions:
- **Coherence:** Does the continuation read like a plausible sentence or dialogue?
- **Contextual consistency:** Does it continue the seed's topic/style?
- **Diversity:** Are the generated words and structures varied?
- **Repetition:** Does the model loop on the same word or phrase?
- **Temperature:** Compare 0.5, 1.0, and 1.5 for the same seed/model.


In [ ]:
# Optional compact table for manually recording observations after reviewing samples.
qualitative_template = pd.DataFrame([
    {"model": m, "temperature": t, "coherence_1_to_5": "", 
     "context_1_to_5": "", "diversity_1_to_5": "", "repetition_1_to_5": "",
     "notes_from_observed_samples": ""}
    for m in MODEL_NAMES for t in TEMPERATURES
])
qualitative_template.to_csv("qualitative_review_template.csv", index=False)
qualitative_template.head()


## 6. Automatically summarize the measured results

The best model by validation perplexity is selected as the quantitative winner. The final conclusion should also discuss generation samples, because the lowest perplexity does not automatically guarantee the most engaging text.


In [ ]:
best = results.iloc[0]
print("Best validation perplexity:", best["model"])
print(f"Validation loss: {best['loss']:.4f}")
print(f"Validation perplexity: {best['perplexity']:.2f}")
print(f"Validation accuracy: {best['accuracy']:.4f}")
print(f"Training time (seconds): {best['training_time_seconds']:.2f}")

print("\\nTemperature interpretation:")
print("- 0.5: generally more conservative and repetitive; often more locally predictable.")
print("- 1.0: balanced baseline between high-probability and less likely tokens.")
print("- 1.5: generally more varied but more prone to odd or incoherent continuations.")
print("\\nWrite your final model conclusion after reviewing the generated samples and metrics.")


## 7. Limitations

1. This is a compact experiment with a limited vocabulary and token budget, not a state-of-the-art language model.
2. Validation is a held-out contiguous tail of the corpus, so style or character distribution may differ from the training prefix.
3. Bidirectional RNN reads both directions inside each already-observed context window. It is suitable for next-word prediction after a known context, but it is not a causal architecture for token-by-token hidden-state streaming.
4. Sampling is stochastic; generated examples may differ unless the random sampling state is also controlled.
5. Training time and exact metrics depend on hardware and runtime settings.
